# Holdings Search Tool Test

This notebook demonstrates the HoldingsSearchTool, which finds funds that invest in specific companies or assets using fuzzy matching with Levenshtein distance.

In [ ]:
import sys

from frontier_challenge.tools import HoldingsSearchTool
from frontier_challenge.tools.holdings_tool.models import HoldingsSearchCriteria
import pandas as pd
import asyncio

## Initialize the Tool

In [ ]:
# Initialize the holdings search tool
tool = HoldingsSearchTool(db_path="../data/br_funds.db")

# View the schema
print(tool.view_schema)

## Example 1: Simple Company Name Search (Fuzzy Matching)

In [ ]:
# Search for funds holding Petrobras
result = await tool.search_holdings(company_name="Petrobras")

print(f"Success: {result.success}")
print(f"Search Method: {result.search_method}")
print(f"Total Results: {result.total_count}")
print(f"Unique Funds: {result.unique_funds_count}")
print(f"Execution Time: {result.execution_time_ms:.2f}ms")
print(f"\nSQL Query:\n{result.sql_query}")

In [ ]:
# Display results as DataFrame
if result.holdings:
    df = pd.DataFrame([h.dict() for h in result.holdings[:20]])
    print(f"\nTop 20 Holdings:")
    display(df[['legal_name', 'asset_name', 'issuer_name', 'portfolio_weight_pct', 'position_value']])
elif result.fund_summaries:
    df = pd.DataFrame([s.dict() for s in result.fund_summaries[:20]])
    print(f"\nTop 20 Funds:")
    display(df[['legal_name', 'asset_name', 'issuer_name', 'portfolio_weight_pct', 'position_value']])

## Example 2: Natural Language Query

In [ ]:
# Natural language query
result = await tool.search_holdings(query="Funds that invest in Vale")

print(f"Success: {result.success}")
print(f"Total Results: {result.total_count}")
print(f"Unique Funds: {result.unique_funds_count}")

if result.fund_summaries:
    df = pd.DataFrame([s.dict() for s in result.fund_summaries[:15]])
    display(df[['legal_name', 'asset_name', 'portfolio_weight_pct']])

## Example 3: Structured Search with Filters

In [ ]:
# Search for funds holding a specific company with additional filters
criteria = HoldingsSearchCriteria(
    company_name="Itau",
    use_fuzzy_match=True,
    min_similarity=0.7,
    asset_class="EQUITY",
    min_weight=1.0,  # At least 1% portfolio weight
    group_by_fund=True,
    sort_by="portfolio_weight_pct",
    sort_descending=True,
    limit=20
)

result = await tool.search_holdings(criteria=criteria)

print(f"Success: {result.success}")
print(f"Search Method: {result.search_method}")
print(f"Total Results: {result.total_count}")

if result.fund_summaries:
    df = pd.DataFrame([s.dict() for s in result.fund_summaries])
    display(df[['legal_name', 'asset_name', 'portfolio_weight_pct', 'position_value']])

## Example 4: Search by Asset Class

In [ ]:
# Find funds holding Brazilian government bonds
criteria = HoldingsSearchCriteria(
    issuer_name="Tesouro",
    use_fuzzy_match=True,
    min_similarity=0.6,
    asset_class="FIXED_INCOME",
    asset_country="BRA",
    min_weight=5.0,
    group_by_fund=True,
    limit=30
)

result = await tool.search_holdings(criteria=criteria)

print(f"Success: {result.success}")
print(f"Unique Funds: {result.unique_funds_count}")

if result.fund_summaries:
    df = pd.DataFrame([s.dict() for s in result.fund_summaries])
    display(df[['legal_name', 'asset_name', 'issuer_name', 'portfolio_weight_pct']])

## Example 5: Detailed Holdings (Not Grouped)

In [ ]:
# Get all holdings of a specific asset across all funds
criteria = HoldingsSearchCriteria(
    company_name="Banco do Brasil",
    use_fuzzy_match=True,
    min_similarity=0.65,
    group_by_fund=False,  # Get all positions, not just top per fund
    sort_by="portfolio_weight_pct",
    sort_descending=True,
    limit=50
)

result = await tool.search_holdings(criteria=criteria)

print(f"Success: {result.success}")
print(f"Total Holdings: {result.total_count}")
print(f"Unique Funds: {result.unique_funds_count}")

if result.holdings:
    df = pd.DataFrame([h.dict() for h in result.holdings[:25]])
    display(df[[
        'legal_name', 
        'asset_name', 
        'asset_class',
        'portfolio_weight_pct', 
        'position_value',
        'asset_country'
    ]])

## Example 6: International Assets

In [ ]:
# Find funds holding US assets
criteria = HoldingsSearchCriteria(
    asset_country="USA",
    min_weight=2.0,
    group_by_fund=True,
    sort_by="portfolio_weight_pct",
    sort_descending=True,
    limit=30
)

result = await tool.search_holdings(criteria=criteria)

print(f"Success: {result.success}")
print(f"Unique Funds with US exposure: {result.unique_funds_count}")

if result.fund_summaries:
    df = pd.DataFrame([s.dict() for s in result.fund_summaries])
    display(df[['legal_name', 'asset_name', 'portfolio_weight_pct', 'position_value']])

## Example 7: Test Fuzzy Matching Threshold

In [ ]:
# Test different similarity thresholds
search_term = "Petro"  # Abbreviated search

for similarity in [0.5, 0.6, 0.7, 0.8]:
    criteria = HoldingsSearchCriteria(
        company_name=search_term,
        use_fuzzy_match=True,
        min_similarity=similarity,
        group_by_fund=True,
        limit=10
    )
    
    result = await tool.search_holdings(criteria=criteria)
    
    print(f"\nSimilarity threshold {similarity}: {result.unique_funds_count} funds found")
    
    if result.fund_summaries and len(result.fund_summaries) > 0:
        print(f"  Top match: {result.fund_summaries[0].asset_name}")

## Performance Test

In [ ]:
import time

# Test performance of different search types
test_cases = [
    {"name": "Exact match", "criteria": HoldingsSearchCriteria(company_name="Petrobras", use_fuzzy_match=False, group_by_fund=True)},
    {"name": "Fuzzy match 0.6", "criteria": HoldingsSearchCriteria(company_name="Petrobras", use_fuzzy_match=True, min_similarity=0.6, group_by_fund=True)},
    {"name": "Fuzzy match 0.8", "criteria": HoldingsSearchCriteria(company_name="Petrobras", use_fuzzy_match=True, min_similarity=0.8, group_by_fund=True)},
]

for test in test_cases:
    result = await tool.search_holdings(criteria=test["criteria"])
    print(f"{test['name']:20} - {result.execution_time_ms:6.2f}ms - {result.unique_funds_count:3} funds")